# 🎬 أيمن الشربيني - صانع الفيديو
## اضغط ▶️ مرة واحدة فقط

In [ ]:
%%capture
!pip install -q diffusers transformers accelerate torchvision pillow edge-tts safetensors
!apt-get install -qq ffmpeg
print("تم التثبيت ✅")

In [ ]:
import torch
from PIL import Image
import asyncio
import subprocess
import os
from pathlib import Path
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler

print("جارٍ تحميل النموذج (مرة واحدة)...")

scheduler = EulerDiscreteScheduler.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", subfolder="scheduler")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    scheduler=scheduler,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
pipe = pipe.to("cuda")
pipe.enable_vae_slicing()
print("النموذج جاهز ✅")

---
## ✏️ عدّل النص هنا
غيّر الوصف في الخلية اللي تحت، وبعدين شغلها ▶️

In [ ]:
import edge_tts
import asyncio
import subprocess
from pathlib import Path

# ===== التعديلات هنا =====
PROMPT = "منظر طبيعي مع جبال وبحيرة وغروب الشمس"
VOICE = "ar-SA-HamedNeural"
OUTPUT = "فيديو_1"
# =========================

print(f"🎨 توليد صورة من: {PROMPT}")
image = pipe(
    prompt=PROMPT,
    negative_prompt="blurry, low quality, distorted, watermark, text",
    num_inference_steps=25,
    guidance_scale=7.5,
    width=1024,
    height=576,
).images[0]

image.save(f"/content/{OUTPUT}.png")
print("✅ الصورة جاهزة")
display(image.resize((512, 288)))

In [ ]:
import edge_tts
import asyncio
import subprocess
from pathlib import Path

async def tts(text, voice, path):
    com = edge_tts.Communicate(text, voice)
    await com.save(path)

print("🔊 توليد الصوت...")
asyncio.run(tts(PROMPT, VOICE, f"/content/{OUTPUT}.mp3"))
print("✅ الصوت جاهز")

In [ ]:
from pathlib import Path

print("🎬 إنشاء الفيديو...")

audio_path = f"/content/{OUTPUT}.mp3"
img_path = f"/content/{OUTPUT}.png"
video_path = f"/content/{OUTPUT}.mp4"

# مدة الصوت
result = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", audio_path],
    capture_output=True, text=True)
duration = float(result.stdout.strip()) if result.stdout else 10

cmd = [
    "ffmpeg", "-y",
    "-loop", "1",
    "-i", img_path,
    "-i", audio_path,
    "-c:v", "libx264",
    "-t", str(duration),
    "-pix_fmt", "yuv420p",
    "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
    "-c:a", "aac",
    "-shortest",
    video_path,
]
subprocess.run(cmd, check=True, capture_output=True)
print(f"✅ الفيديو جاهز: {video_path}")
from IPython.display import Video
display(Video(video_path, width=400))